# Stage 3.0: Continuous Batching Explained

## Problem Statement
Static batching forces all requests to wait for the longest one to complete, leading to significant GPU idle time and poor throughput. How can we dynamically add and remove requests to maximize GPU utilization?

## What You'll Learn
- Static vs continuous batching comparison
- Iteration-level scheduling concept
- Why continuous batching achieves 3-10x higher throughput
- Dynamic request addition/removal mechanics
- Implementation considerations for production
- Real-world performance analysis

## Prerequisites
- Understanding of KV cache (notebook 10)
- Familiarity with static batching (notebook 12)
- Basic knowledge of inference serving

---

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import List, Optional
from collections import deque
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## Part 1: The Problem with Static Batching

### Static Batching Timeline

In static batching, all requests must:
1. Start together as a batch
2. Wait for the longest request to complete
3. Cannot add new requests until entire batch finishes

**Example:**
```
Request A: Generate 10 tokens
Request B: Generate 100 tokens  
Request C: Generate 50 tokens

Static Batching Timeline:
|----A----|                          (finishes at t=10, waits until t=100)
|--------------------B---------------| (longest, determines batch completion)
|----------C----------|               (finishes at t=50, waits until t=100)
                                     New batch starts here |

GPU Idle Time: 60% (waiting for slowest request)
Wasted Compute: Padding and waiting
```

In [ ]:
@dataclass
class Request:
    """Represents a single inference request"""
    request_id: int
    num_tokens_to_generate: int
    tokens_generated: int = 0
    start_time: float = 0.0
    finish_time: float = 0.0
    
    def is_finished(self) -> bool:
        return self.tokens_generated >= self.num_tokens_to_generate
    
    def latency(self) -> float:
        if self.finish_time == 0.0:
            return 0.0
        return self.finish_time - self.start_time


def simulate_static_batching(requests: List[Request], batch_size: int = 4):
    """
    Simulate static batching where all requests in a batch must finish before
    the next batch can start.
    """
    results = []
    time_per_token = 0.01  # 10ms per token
    current_time = 0.0
    
    # Process in batches
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        
        # All requests start at the same time
        for req in batch:
            req.start_time = current_time
        
        # Find longest request in batch
        max_tokens = max(req.num_tokens_to_generate for req in batch)
        
        # All requests must wait for longest one
        batch_duration = max_tokens * time_per_token
        
        # Mark all as finished at same time
        for req in batch:
            req.tokens_generated = req.num_tokens_to_generate
            req.finish_time = current_time + batch_duration
        
        current_time += batch_duration
        results.extend(batch)
    
    return results, current_time


# Create sample requests with varying lengths
static_requests = [
    Request(0, 10),   # Short
    Request(1, 100),  # Long
    Request(2, 50),   # Medium
    Request(3, 30),   # Medium-short
    Request(4, 80),   # Long
    Request(5, 20),   # Short
    Request(6, 60),   # Medium
    Request(7, 15),   # Short
]

static_results, static_total_time = simulate_static_batching(static_requests.copy(), batch_size=4)

print("\n=== STATIC BATCHING RESULTS ===")
print(f"Total time: {static_total_time:.2f}s")
print(f"Average latency: {np.mean([r.latency() for r in static_results]):.2f}s")
print(f"P95 latency: {np.percentile([r.latency() for r in static_results], 95):.2f}s")
print(f"\nPer-request breakdown:")
for req in static_results:
    print(f"  Request {req.request_id}: {req.num_tokens_to_generate} tokens, "
          f"latency={req.latency():.2f}s")

---

## Part 2: Continuous Batching - Iteration-Level Scheduling

### The Key Innovation

**Continuous batching** (also called **iteration-level scheduling**) allows:
1. **Add new requests immediately** when there's available memory/compute
2. **Remove finished requests immediately** without waiting for others
3. **Process at the iteration level** (per-token, not per-batch)

### How It Works

```python
# Traditional Static Batching
while True:
    batch = wait_for_batch(batch_size)  # Wait for full batch
    results = process_batch(batch)      # Process until all finish
    return_results(results)             # Return all together

# Continuous Batching
active_requests = []
while True:
    # Add new requests if space available
    while has_capacity() and new_requests:
        active_requests.append(new_requests.pop())
    
    # Generate one token for ALL active requests
    generate_one_token(active_requests)
    
    # Remove finished requests immediately
    active_requests = [r for r in active_requests if not r.finished()]
```

### Timeline Comparison

```
Continuous Batching Timeline:
|----A----|                          (removed immediately when done)
           |--D--|                   (added when A finishes)
|--------------------B---------------|
|----------C----------|
                      |-----E--------|  (added when C finishes)

GPU Idle Time: <5% (nearly always processing)
Throughput: 3-10x higher than static batching
```

In [ ]:
def simulate_continuous_batching(requests: List[Request], max_batch_size: int = 4):
    """
    Simulate continuous batching with iteration-level scheduling.
    Requests can be added/removed at any iteration (token).
    """
    time_per_token = 0.01  # 10ms per token
    current_time = 0.0
    
    # Queue of pending requests
    pending_queue = deque(requests)
    
    # Currently active requests being processed
    active_requests = []
    
    # Completed requests
    completed = []
    
    iteration = 0
    
    while pending_queue or active_requests:
        iteration += 1
        
        # Step 1: Add new requests if we have capacity
        while len(active_requests) < max_batch_size and pending_queue:
            new_request = pending_queue.popleft()
            new_request.start_time = current_time
            active_requests.append(new_request)
        
        if not active_requests:
            break
        
        # Step 2: Generate one token for all active requests
        for req in active_requests:
            req.tokens_generated += 1
        
        current_time += time_per_token
        
        # Step 3: Remove finished requests immediately
        still_active = []
        for req in active_requests:
            if req.is_finished():
                req.finish_time = current_time
                completed.append(req)
            else:
                still_active.append(req)
        
        active_requests = still_active
    
    return completed, current_time


# Use same requests for fair comparison
continuous_requests = [
    Request(0, 10),
    Request(1, 100),
    Request(2, 50),
    Request(3, 30),
    Request(4, 80),
    Request(5, 20),
    Request(6, 60),
    Request(7, 15),
]

continuous_results, continuous_total_time = simulate_continuous_batching(
    continuous_requests.copy(), max_batch_size=4
)

print("\n=== CONTINUOUS BATCHING RESULTS ===")
print(f"Total time: {continuous_total_time:.2f}s")
print(f"Average latency: {np.mean([r.latency() for r in continuous_results]):.2f}s")
print(f"P95 latency: {np.percentile([r.latency() for r in continuous_results], 95):.2f}s")
print(f"\nPer-request breakdown:")
for req in continuous_results:
    print(f"  Request {req.request_id}: {req.num_tokens_to_generate} tokens, "
          f"latency={req.latency():.2f}s")

---

## Part 3: Performance Comparison

Let's compare static vs continuous batching across multiple metrics.

In [ ]:
# Direct comparison
print("\n" + "="*60)
print("STATIC vs CONTINUOUS BATCHING COMPARISON")
print("="*60)

print(f"\nTotal Completion Time:")
print(f"  Static:     {static_total_time:.2f}s")
print(f"  Continuous: {continuous_total_time:.2f}s")
print(f"  Speedup:    {static_total_time / continuous_total_time:.2f}x")

static_avg_latency = np.mean([r.latency() for r in static_results])
continuous_avg_latency = np.mean([r.latency() for r in continuous_results])

print(f"\nAverage Latency per Request:")
print(f"  Static:     {static_avg_latency:.2f}s")
print(f"  Continuous: {continuous_avg_latency:.2f}s")
print(f"  Improvement: {static_avg_latency / continuous_avg_latency:.2f}x")

static_p95 = np.percentile([r.latency() for r in static_results], 95)
continuous_p95 = np.percentile([r.latency() for r in continuous_results], 95)

print(f"\nP95 Latency:")
print(f"  Static:     {static_p95:.2f}s")
print(f"  Continuous: {continuous_p95:.2f}s")
print(f"  Improvement: {static_p95 / continuous_p95:.2f}x")

# Throughput (tokens/sec)
total_tokens = sum(r.num_tokens_to_generate for r in static_requests)
static_throughput = total_tokens / static_total_time
continuous_throughput = total_tokens / continuous_total_time

print(f"\nThroughput (tokens/sec):")
print(f"  Static:     {static_throughput:.1f}")
print(f"  Continuous: {continuous_throughput:.1f}")
print(f"  Improvement: {continuous_throughput / static_throughput:.2f}x")

---

## Part 4: Visual Comparison - Timeline Charts

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Static batching timeline
for req in static_results:
    ax1.barh(req.request_id, 
            req.latency(), 
            left=req.start_time,
            height=0.8,
            alpha=0.7,
            label=f"Req {req.request_id}" if req.request_id < 4 else "")
    # Show actual work vs waiting
    actual_work = req.num_tokens_to_generate * 0.01
    ax1.barh(req.request_id,
            actual_work,
            left=req.start_time,
            height=0.8,
            color='green',
            alpha=0.9)

ax1.set_xlabel('Time (seconds)', fontsize=12)
ax1.set_ylabel('Request ID', fontsize=12)
ax1.set_title('Static Batching: Requests Wait for Slowest in Batch', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, max(static_total_time, continuous_total_time) * 1.05)

# Continuous batching timeline
for req in continuous_results:
    ax2.barh(req.request_id, 
            req.latency(), 
            left=req.start_time,
            height=0.8,
            color='green',
            alpha=0.9,
            label=f"Req {req.request_id}" if req.request_id < 4 else "")

ax2.set_xlabel('Time (seconds)', fontsize=12)
ax2.set_ylabel('Request ID', fontsize=12)
ax2.set_title('Continuous Batching: Requests Finish Immediately When Done', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, max(static_total_time, continuous_total_time) * 1.05)

# Add legend for static batching
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.9, label='Actual Work'),
    Patch(facecolor='blue', alpha=0.3, label='Waiting for Batch')
]
ax1.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('continuous_vs_static_batching.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'continuous_vs_static_batching.png'")

---

## Part 5: GPU Utilization Analysis

One of the key benefits of continuous batching is higher GPU utilization.

In [ ]:
def calculate_gpu_utilization(results, total_time, batch_size):
    """
    Calculate GPU utilization over time.
    GPU utilization = (actual work done) / (max possible work)
    """
    total_tokens_generated = sum(r.num_tokens_to_generate for r in results)
    max_possible_tokens = total_time / 0.01 * batch_size
    utilization = total_tokens_generated / max_possible_tokens
    return utilization * 100

static_util = calculate_gpu_utilization(static_results, static_total_time, 4)
continuous_util = calculate_gpu_utilization(continuous_results, continuous_total_time, 4)

print("\n=== GPU UTILIZATION ===")
print(f"Static Batching:     {static_util:.1f}%")
print(f"Continuous Batching: {continuous_util:.1f}%")
print(f"\nImprovement: {continuous_util - static_util:.1f} percentage points")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['Static\nBatching', 'Continuous\nBatching']
utilizations = [static_util, continuous_util]
colors = ['#ff6b6b', '#4ecdc4']

bars = ax.bar(methods, utilizations, color=colors, alpha=0.8, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('GPU Utilization (%)', fontsize=12)
ax.set_title('GPU Utilization: Static vs Continuous Batching', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=100, color='gray', linestyle='--', linewidth=1, label='Maximum')

plt.tight_layout()
plt.savefig('gpu_utilization_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

---

## Part 6: Implementation Considerations

### Memory Management

Continuous batching requires careful memory management:

```python
# Challenge: Variable batch sizes mean variable memory usage
# Solution: Pre-allocate memory pools or use PagedAttention (see notebook 31)

class MemoryManager:
    def __init__(self, max_memory_gb: float):
        self.max_memory = max_memory_gb * 1e9
        self.current_usage = 0
    
    def can_add_request(self, kv_cache_size: int) -> bool:
        return self.current_usage + kv_cache_size <= self.max_memory
    
    def allocate(self, kv_cache_size: int):
        if self.can_add_request(kv_cache_size):
            self.current_usage += kv_cache_size
            return True
        return False
    
    def free(self, kv_cache_size: int):
        self.current_usage -= kv_cache_size
```

### Request Scheduling

```python
# Different scheduling policies:

# 1. FIFO (First In First Out) - Fair but may not be optimal
def fifo_schedule(queue, active, max_batch):
    while len(active) < max_batch and queue:
        active.append(queue.pop(0))

# 2. Shortest Job First (SJF) - Minimize average latency
def sjf_schedule(queue, active, max_batch):
    queue.sort(key=lambda r: r.estimated_tokens)
    while len(active) < max_batch and queue:
        active.append(queue.pop(0))

# 3. Priority-based - Different SLAs for different users
def priority_schedule(queue, active, max_batch):
    queue.sort(key=lambda r: r.priority, reverse=True)
    while len(active) < max_batch and queue:
        active.append(queue.pop(0))
```

### KV Cache Management

```python
# With continuous batching, KV cache becomes more complex:

class KVCache:
    def __init__(self, max_batch_size, max_seq_len, num_layers, hidden_dim):
        # Allocate maximum possible cache upfront
        self.cache_k = torch.zeros(
            (max_batch_size, num_layers, max_seq_len, hidden_dim)
        )
        self.cache_v = torch.zeros(
            (max_batch_size, num_layers, max_seq_len, hidden_dim)
        )
        self.active_slots = set()
    
    def allocate_slot(self) -> Optional[int]:
        for i in range(self.cache_k.size(0)):
            if i not in self.active_slots:
                self.active_slots.add(i)
                return i
        return None
    
    def free_slot(self, slot: int):
        self.active_slots.remove(slot)
        # Zero out the cache for this slot
        self.cache_k[slot].zero_()
        self.cache_v[slot].zero_()
```

---

## Part 7: Scaling Analysis - Varying Request Arrival Rates

In [ ]:
def generate_random_requests(num_requests: int, seed: int = 42) -> List[Request]:
    """Generate requests with random token counts"""
    np.random.seed(seed)
    requests = []
    for i in range(num_requests):
        # Token counts follow realistic distribution (most are short, some are long)
        tokens = int(np.random.lognormal(3.5, 0.8))  # Mean ~50, long tail
        tokens = max(10, min(tokens, 200))  # Clamp between 10-200
        requests.append(Request(i, tokens))
    return requests

# Benchmark across different scales
request_counts = [10, 20, 50, 100]
batch_size = 8

static_times = []
continuous_times = []
speedups = []

for count in request_counts:
    requests = generate_random_requests(count)
    
    # Static batching
    static_reqs = [Request(r.request_id, r.num_tokens_to_generate) for r in requests]
    _, static_time = simulate_static_batching(static_reqs, batch_size)
    static_times.append(static_time)
    
    # Continuous batching
    continuous_reqs = [Request(r.request_id, r.num_tokens_to_generate) for r in requests]
    _, continuous_time = simulate_continuous_batching(continuous_reqs, batch_size)
    continuous_times.append(continuous_time)
    
    speedups.append(static_time / continuous_time)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Total time comparison
x = np.arange(len(request_counts))
width = 0.35

ax1.bar(x - width/2, static_times, width, label='Static Batching', color='#ff6b6b', alpha=0.8)
ax1.bar(x + width/2, continuous_times, width, label='Continuous Batching', color='#4ecdc4', alpha=0.8)

ax1.set_xlabel('Number of Requests', fontsize=12)
ax1.set_ylabel('Total Time (seconds)', fontsize=12)
ax1.set_title('Completion Time vs Number of Requests', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(request_counts)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Speedup
ax2.plot(request_counts, speedups, marker='o', linewidth=2, markersize=10, color='#9b59b6')
ax2.fill_between(request_counts, 1, speedups, alpha=0.3, color='#9b59b6')
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1, label='No speedup')

ax2.set_xlabel('Number of Requests', fontsize=12)
ax2.set_ylabel('Speedup (Static / Continuous)', fontsize=12)
ax2.set_title('Continuous Batching Speedup', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

for i, speedup in enumerate(speedups):
    ax2.text(request_counts[i], speedup, f'{speedup:.2f}x', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('scaling_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== SCALING ANALYSIS ===")
for i, count in enumerate(request_counts):
    print(f"\n{count} requests:")
    print(f"  Static:     {static_times[i]:.2f}s")
    print(f"  Continuous: {continuous_times[i]:.2f}s")
    print(f"  Speedup:    {speedups[i]:.2f}x")

---

## Part 8: Production Implementation Example

Here's a simplified but production-ready continuous batching engine:

In [ ]:
import threading
from queue import Queue
from typing import Callable

class ContinuousBatchingEngine:
    """
    Production-ready continuous batching engine.
    This is a simplified version - real implementations use vLLM or TensorRT-LLM.
    """
    
    def __init__(self, 
                 max_batch_size: int = 32,
                 max_seq_length: int = 2048,
                 model_inference_fn: Optional[Callable] = None):
        self.max_batch_size = max_batch_size
        self.max_seq_length = max_seq_length
        self.model_inference_fn = model_inference_fn
        
        # Request queues
        self.pending_queue = Queue()
        self.active_requests = []
        
        # Metrics
        self.total_requests_processed = 0
        self.total_tokens_generated = 0
        
        self.running = False
        self.worker_thread = None
    
    def submit_request(self, request: Request):
        """Submit a new request for processing"""
        self.pending_queue.put(request)
    
    def _can_add_request(self) -> bool:
        """Check if we have capacity for a new request"""
        if len(self.active_requests) >= self.max_batch_size:
            return False
        
        # Check memory constraints (simplified)
        # In reality, need to check actual GPU memory
        max_seq = max([r.tokens_generated for r in self.active_requests], default=0)
        return max_seq < self.max_seq_length
    
    def _process_iteration(self):
        """
        Process one iteration:
        1. Add new requests if capacity available
        2. Generate one token for all active requests
        3. Remove finished requests
        """
        # Step 1: Add new requests
        while self._can_add_request() and not self.pending_queue.empty():
            request = self.pending_queue.get()
            request.start_time = time.time()
            self.active_requests.append(request)
        
        if not self.active_requests:
            return False  # No work to do
        
        # Step 2: Generate tokens (in real implementation, call model)
        if self.model_inference_fn:
            # Call actual model inference
            self.model_inference_fn(self.active_requests)
        else:
            # Simulate token generation
            time.sleep(0.01)  # 10ms per token
            for req in self.active_requests:
                req.tokens_generated += 1
        
        # Step 3: Remove finished requests
        still_active = []
        for req in self.active_requests:
            if req.is_finished():
                req.finish_time = time.time()
                self.total_requests_processed += 1
                self.total_tokens_generated += req.num_tokens_to_generate
            else:
                still_active.append(req)
        
        self.active_requests = still_active
        return True
    
    def start(self):
        """Start the continuous batching engine"""
        self.running = True
        self.worker_thread = threading.Thread(target=self._worker_loop)
        self.worker_thread.start()
    
    def stop(self):
        """Stop the engine"""
        self.running = False
        if self.worker_thread:
            self.worker_thread.join()
    
    def _worker_loop(self):
        """Main worker loop"""
        while self.running:
            has_work = self._process_iteration()
            if not has_work:
                time.sleep(0.001)  # Small sleep if no work
    
    def get_metrics(self):
        """Get current metrics"""
        return {
            'total_requests_processed': self.total_requests_processed,
            'total_tokens_generated': self.total_tokens_generated,
            'active_batch_size': len(self.active_requests),
            'pending_requests': self.pending_queue.qsize()
        }


# Example usage
print("\n=== CONTINUOUS BATCHING ENGINE DEMO ===")

engine = ContinuousBatchingEngine(max_batch_size=4)
engine.start()

# Submit requests
test_requests = generate_random_requests(10)
for req in test_requests:
    engine.submit_request(req)

# Wait for completion
print("\nProcessing requests...")
while engine.pending_queue.qsize() > 0 or len(engine.active_requests) > 0:
    metrics = engine.get_metrics()
    print(f"  Active: {metrics['active_batch_size']}, "
          f"Pending: {metrics['pending_requests']}, "
          f"Completed: {metrics['total_requests_processed']}/10")
    time.sleep(0.1)

engine.stop()

final_metrics = engine.get_metrics()
print(f"\nFinal metrics:")
print(f"  Total requests processed: {final_metrics['total_requests_processed']}")
print(f"  Total tokens generated: {final_metrics['total_tokens_generated']}")

---

## Part 9: Key Takeaways

### Why Continuous Batching is Superior

1. **Higher Throughput (3-10x)**
   - No waiting for slowest request
   - Immediately fill freed slots with new requests
   - Better GPU utilization (60-80% vs 30-50%)

2. **Lower Latency**
   - Requests finish as soon as they're done
   - No artificial waiting
   - Better P95/P99 latency metrics

3. **Better Resource Utilization**
   - Memory used more efficiently
   - GPU rarely idle
   - Scales better with load

4. **Fairness**
   - Short requests don't wait for long ones
   - More predictable latency
   - Better user experience

### When to Use Each Approach

**Static Batching:**
- Simple to implement
- Good for offline batch processing
- When all requests similar length
- Research/development environments

**Continuous Batching:**
- Production serving (always recommended)
- Multi-user applications
- Variable request lengths
- When throughput matters

### Production Frameworks

Don't implement from scratch! Use:
- **vLLM** (recommended) - PagedAttention + continuous batching
- **TensorRT-LLM** - NVIDIA's optimized solution
- **Text Generation Inference (TGI)** - HuggingFace's solution

### Memory Considerations

Continuous batching works best with:
- **PagedAttention** (notebook 31) - Eliminates memory fragmentation
- **Prefix caching** (notebook 33) - Share common prefixes
- **Multi-Query Attention** - Reduce KV cache size

---

## References & Next Steps

### Papers
- "Orca: A Distributed Serving System for Transformer-Based Generative Models" (Yu et al., 2022)
- "Efficient Memory Management for Large Language Model Serving with PagedAttention" (Kwon et al., 2023)

### Related Notebooks
- **Notebook 12**: Static batching fundamentals
- **Notebook 31**: PagedAttention (enables efficient continuous batching)
- **Notebook 33**: Prefix caching (optimizes multi-turn conversations)
- **Notebook 34**: Multi-GPU serving strategies

### Production Resources
- vLLM documentation: https://docs.vllm.ai/
- TensorRT-LLM: https://github.com/NVIDIA/TensorRT-LLM
- LLM Inference Roadmap: `LLM_INFERENCE_ROADMAP.md`

---

## Summary

Continuous batching is the **most important optimization for production serving**, providing:
- 3-10x higher throughput than static batching
- Lower and more predictable latency
- Better GPU utilization (60-80% vs 30-50%)

**Key principle:** Process at the iteration level (per-token), not batch level.

Combined with PagedAttention (next notebook), continuous batching enables serving at scale.

**Next:** Notebook 31 - PagedAttention (the memory management breakthrough that makes continuous batching practical)